# German Skincare RAG Chatbot
* The flow is: parse a user's skin profile, search embeddings with FAISS, rerank products with skincare rules, compress the context to fit a token budget and then generate the final answer through LM Studio
* 3 main sources: csv of Rossmann skin products `products_cleaned.csv`, json file of ingredient knowledge `paulas_choice_ingredients.json`, a directory of routine images from instagram influencer xskincare `instagram_downloads/xskincare_CvUO2zWK_N-`
* Since I had problems with the token budget, I had to decompress the input and answer



In [1]:
import json
import re
from pathlib import Path
import faiss
import pandas as pd
from IPython.display import Markdown, display
from openai import OpenAI
from PIL import Image
import pytesseract
from sentence_transformers import SentenceTransformer

c:\Users\julia\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data sources and configuration

In [2]:
# paths to data sources
products_csv = Path('products_cleaned.csv')
INGREDIENTS_JSON = Path('paulas_choice_ingredients.json')

insta_dir = Path('instagram_downloads/xskincare_CvUO2zWK_N-')
ROUTINE_IMAGE_PATHS = [
    insta_dir / '2023-07-30_09-15-37_UTC_1.jpg',
    insta_dir / '2023-07-30_09-15-37_UTC_2.jpg',
    insta_dir / '2023-07-30_09-15-37_UTC_3.jpg',
    insta_dir / '2023-07-30_09-15-37_UTC_4.jpg',
    insta_dir / '2023-07-30_09-15-37_UTC_5.jpg',
    insta_dir / '2023-07-30_09-15-37_UTC_6.jpg',
    insta_dir / '2023-07-30_09-15-37_UTC_7.jpg',
]

# embedding model used to covert text into vectors for semantic search
embed_model = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

# local LLM server configuration for generation
lmstudio_base_url = 'http://localhost:1234/v1'
lmstudio_api_key = 'lm-studio'
lmstudio_model = 'qwen2.5-7b-instruct'

# token budgeting for prompt construction
# input context
max_context_tokens = 3200
# final answer
target_response_tokens = 500
# extra room to avoid exceeding the limit
token_safety_margin = 250

# how many retrieved items to include from each source
max_ingredient_hits = 2
max_routine_hits = 1
max_product_hits = 3

# truncation limits for compacting long text fields
max_ingredient_text_len = 180
max_routine_text_len = 120
max_product_instruction_len = 100
max_product_ingredients_len = 120
max_product_warnings_len = 80

## Helper functions to prepare text

In [3]:
def clean_text(text):
    """This function standardizes text input. It converts everything to a string."""
    if pd.isna(text):
        return ''
    text = str(text)
    # multiple spaces into a single space
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def normalize(text):
    """Lowercase version of clean_text()."""
    return clean_text(text).lower()


def make_doc(source_type, row_id, text, meta=None, title=''):
    """This function creates a consistent document dictionary."""
    return {
        'source_type': source_type,
        'row_id': int(row_id),
        'title': clean_text(title),
        'text': clean_text(text),
        'meta': meta or {},
    }


def shorten(text, max_len=220):
    """This function trims long text to a readable preview."""
    text = clean_text(text)
    if len(text) <= max_len:
        return text
    return text[:max_len].rstrip() + ' ...'

## Functions to load 3 data sources and convert them into a unified document format for retrieval

In [4]:
def load_products(csv_path: Path):
    """This function reads the product csv into a dataframe and checks that all required columns exist.
    It returns a dataframe with the original table and a list of searchable product documents.
    """
    df = pd.read_csv(csv_path)
    expected = {'href', 'name', 'brand', 'price', 'product_details', 'instruction', 'ingredients', 'warnings'}
    
    # get missing columns and raise an error
    missing = expected - set(df.columns)
    if missing:
        raise ValueError(f'Fehlende CSV-Spalten: {missing}')

    # loop through each row and build a dictionary with cleaned fields,
    # a long formatted text block with readable labels
    # and a document via make_doc
    # df is the original table, docs is a list of searchable product documents
    docs = []
    for idx, row in df.iterrows():
        meta = {
            'href': clean_text(row['href']),
            'name': clean_text(row['name']),
            'brand': clean_text(row['brand']),
            'price': clean_text(row['price']),
            'product_details': clean_text(row['product_details']),
            'instruction': clean_text(row['instruction']),
            'ingredients': clean_text(row['ingredients']),
            'warnings': clean_text(row['warnings']),
        }
        # text string concatenates the important product attributes into one chunk
        # this makes each product easier to embed and retrieve semantically because 
        # the model sees all relevant product context together
        text = f'''Produktname: {meta['name']}
Marke: {meta['brand']}
Preis: {meta['price']}
Beschreibung: {meta['product_details']}
Anwendung: {meta['instruction']}
Inhaltsstoffe: {meta['ingredients']}
Warnhinweise: {meta['warnings']}
Link: {meta['href']}'''.strip()
        docs.append(make_doc('product', idx, text, meta=meta, title=f"{meta['brand']} {meta['name']}"))
    return df, docs


def load_ingredient_knowledge(json_path: Path):
    """This function opens the json file and loads ingredient notes into document objects."""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    docs = []
    # each json item is cleaned and rewritten into a structured text format 
    for i, item in enumerate(data):
        topic = clean_text(item.get('topic', ''))
        ingredient = clean_text(item.get('ingredient', ''))
        text = clean_text(item.get('text', ''))
        source_url = clean_text(item.get('source_url', ''))
        docs.append(make_doc(
            'ingredient_knowledge',
            i,
            f'Thema: {topic}Wirkstoff: {ingredient} Erklärung: {text} Quelle: {source_url}',
            meta={'topic': topic, 'ingredient': ingredient, 'text': text, 'source_url': source_url},
            title=f'{ingredient} - {topic}'
        ))
    return docs


def ocr_image(image_path: Path, lang: str = 'deu+eng'):
    """This function opens an image with PIL and extracts text using Tesseract OCR. """
    image = Image.open(image_path)
    text = pytesseract.image_to_string(image, lang=lang)
    return clean_text(text)


def load_routine_images(image_paths):
    """This function processes the routine screenshots one by one and turns them into documents."""
    docs, missing_files, failed_files = [], [], []
    for i, image_path in enumerate(image_paths):
        # if file does not exist, it is added to missing_files
        if not image_path.exists():
            missing_files.append(str(image_path))
            continue
        try:
            ocr_text = ocr_image(image_path)
            # if OCR returns no text, the file name is added to failed_files
            if not ocr_text:
                failed_files.append(f'{image_path.name} (kein OCR-Text erkannt)')
                continue
            # if OCR works, the image text is stored as a routine_image document with metadata like file name and OCR text
            docs.append(make_doc(
                'routine_image',
                i,
                f'Routinebild-Datei: {image_path.name} OCR-Text: {ocr_text}',
                meta={'file_name': image_path.name, 'ocr_text': ocr_text},
                title=image_path.name
            ))
        except Exception as e:
            failed_files.append(f'{image_path.name} ({e})')
    return docs, missing_files, failed_files


## Functions to build and query a semantic search index over documents

In [5]:
def build_index(docs, model):
    """This function turns all document texts into vector embeddings and stores them in a FAISS index.
    Documents -> embeddings -> searchable vector index"""
    if not docs:
        return None
    # collect the text from each document 
    texts = [d['text'] for d in docs]

    # convert texts into embeddings
    # make vectors unit-length so innter product behaves like cosine similarity
    embeddings = model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
    # create a flat index that uses inner product as the similarity score
    index = faiss.IndexFlatIP(embeddings.shape[1])
    # store all embeddings in the index for later retrieval
    index.add(embeddings.astype('float32'))
    return index

def semantic_search(query, docs, index, model, top_k=5):
    """This function takes a user query and finds the most similar documents in the index.
    Because the embeddings are normalized and the index uses inner product, higher scores mean more semantic similarity between the query and the document."""
    if not docs or index is None:
        return []
    # embed the user query in the same vector space as the documents
    query_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    # ask FAISS for the nearest neighbors
    scores, ids = index.search(query_emb, min(top_k, len(docs)))
    results = []
    # loop through the returned scores and document ids, skip invalid ids and copy each matching document
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        item = docs[idx].copy()
        item['score'] = float(score)
        results.append(item)
    return results

## Functions to turn raw search results into a more skincare-aware ranking
* What did the user say? -> Which products are actually a good match for that skin profile?

In [6]:
def extract_skin_profile(user_text: str):
    """This function reads the user's message and tries to identify skin concers and types with a map."""
    t = normalize(user_text)
    concern_map = {
        'unreinheiten': ['pickel', 'akne', 'unreinheiten'],
        'verstopfte poren': ['verstopfte poren', 'poren', 'mitesser'],
        'trockenheit': ['trocken', 'dehydriert', 'spannt', 'feuchtigkeitsarm'],
        'rötungen': ['rötung', 'rötungen', 'empfindlich', 'sensibel', 'gereizt'],
        'pigmentflecken': ['pigment', 'post-akne', 'flecken', 'ungleichmäßiger hautton'],
        'falten': ['falten', 'linien', 'anti aging', 'anti-aging'],
    }
    skin_type_map = {
        'trockene Haut': ['trockene haut', 'trocken'],
        'fettige Haut': ['fettige haut', 'ölig', 'glänzt'],
        'Mischhaut': ['mischhaut'],
        'sensible Haut': ['sensible haut', 'empfindliche haut', 'sensibel', 'empfindlich'],
    }
    concerns, skin_types = [], []
    # loop through the mappings and check whether any keyword appears in the user text
    # if yes, then matching label is added to the result
    for label, variants in concern_map.items():
        if any(v in t for v in variants):
            concerns.append(label)
    for label, variants in skin_type_map.items():
        if any(v in t for v in variants):
            skin_types.append(label)
    return {'concerns': sorted(set(concerns)), 'skin_types': sorted(set(skin_types))}


def detect_product_type(text: str):
    """This function guesses what kind of skincare product a text describes.
    It returns the first matching product type it finds."""
    t = normalize(text)
    mapping = {
        'cleanser': ['reinigung', 'reinigungs', 'waschgel', 'cleanser', 'mizellen', 'reinigungsöl'],
        'serum': ['serum', 'ampoule', 'ampulle'],
        'moisturizer': ['creme', 'gel', 'fluid', 'nachtpflege', 'tagespflege'],
        'sunscreen': ['lsf', 'spf', 'sonnenschutz', 'uv', 'sun'],
        'mask': ['maske', 'tuchmaske', 'augenpads', 'hydrogel'],
    }
    for product_type, keywords in mapping.items():
        if any(k in t for k in keywords):
            return product_type
    return 'other'


def ingredient_flags(ingredients_text: str):
    """This function looks at a product's ingredient list and turns it into a set of yes/no flags.
    It checks whether certain ingredient families are present."""
    t = normalize(ingredients_text)
    return {
        'has_niacinamide': 'niacinamide' in t,
        'has_hyaluron': 'hyaluron' in t or 'sodium hyaluronate' in t,
        'has_panthenol': 'panthenol' in t,
        'has_ceramide': 'ceramide' in t,
        'has_peptides': 'peptide' in t,
        'has_allantoin': 'allantoin' in t,
        'has_ectoin': 'ectoin' in t,
        'has_glycerin': 'glycerin' in t,
        'has_urea': 'urea' in t,
        'has_bha': 'salicylic acid' in t or 'salicyl' in t,
        'has_aha': 'glycolic acid' in t or 'lactic acid' in t or 'aha' in t,
        'has_vitamin_c': 'ascorb' in t or 'vitamin c' in t,
        'has_retinoid': 'retinol' in t or 'retinal' in t or 'retinyl' in t,
        'has_fragrance': 'parfum' in t or 'fragrance' in t,
        'has_alcohol_denat': 'alcohol denat' in t,
    }


def score_product(doc, profile):
    """This function assigns a skincare relevance score to a product. It starts with the product's 
    existing semantic search score, then adjusts it based on the user's skin profile and the product's ingredients."""
    meta = doc['meta']
    flags = ingredient_flags(meta.get('ingredients', ''))
    product_type = detect_product_type(doc['text'])
    score = doc.get('score', 0.0)
    concerns = profile['concerns']
    skin_types = profile['skin_types']
    if 'verstopfte poren' in concerns or 'unreinheiten' in concerns:
        if flags['has_bha'] or flags['has_niacinamide']:
            score += 0.22
    if 'trockenheit' in concerns:
        if flags['has_hyaluron'] or flags['has_glycerin'] or flags['has_urea'] or flags['has_panthenol']:
            score += 0.22
    if 'rötungen' in concerns:
        if flags['has_panthenol'] or flags['has_allantoin'] or flags['has_ceramide'] or flags['has_ectoin']:
            score += 0.24
    if 'falten' in concerns:
        if flags['has_retinoid'] or flags['has_peptides']:
            score += 0.20
    if 'sensible Haut' in skin_types:
        if flags['has_fragrance']:
            score -= 0.12
        if flags['has_alcohol_denat']:
            score -= 0.15
        if flags['has_retinoid'] or flags['has_aha'] or flags['has_bha']:
            score -= 0.08
    return score, product_type


def rerank_products(product_hits, profile, max_items=8):
    """This function applies score_product() to every product returned from semantic search. For each product:
    - it computes a final_score
    - stores the detected product_type
    - keeps the updated document.
    
    Then it sorts all products by final_score in descending order and keeps only the top max_items"""
    rescored = []
    for doc in product_hits:
        final_score, product_type = score_product(doc, profile)
        item = doc.copy()
        item['final_score'] = final_score
        item['product_type'] = product_type
        rescored.append(item)
    rescored.sort(key=lambda x: x['final_score'], reverse=True)
    return rescored[:max_items]


## Functions to take the ranked products and turn them into a practical weekly skincare routine

In [7]:
def choose_best_product_by_type(products, wanted_type, used_ids=None):
    """This function looks through a list of products and returns the first one that matches the requested product type. """
    used_ids = used_ids or set()
    for p in products:
        unique_id = (p['source_type'], p['row_id'])
        if p.get('product_type') == wanted_type and unique_id not in used_ids:
            return p
    return None


def summarize_product_for_prompt(product):
    """This function makes a compact version of a product for the final prompt. 
    Instead of passing the whole product object, it keeps only the most important fields. """
    if not product:
        return None
    meta = product.get('meta', {})
    return {
        'brand': clean_text(meta.get('brand', '')),
        'name': clean_text(meta.get('name', '')),
        'price': clean_text(meta.get('price', '')),
        'product_type': clean_text(product.get('product_type', '')),
    }


def summarize_weekly_plan(weekly_plan):
    """This function compresses the full weekly routine into a smaller structure."""
    compact_days = []
    for day_entry in weekly_plan:
        morning = []
        evening = []
        # loop through morning and evening steps and replace each full product object with the shorter summary
        for step in day_entry.get('morning', []):
            p = summarize_product_for_prompt(step.get('product'))
            morning.append({'step': step.get('step', ''), 'product': p})
        for step in day_entry.get('evening', []):
            p = summarize_product_for_prompt(step.get('product'))
            evening.append({'step': step.get('step', ''), 'product': p})
        compact_days.append({'day': day_entry.get('day', ''), 'morning': morning, 'evening': evening})
    return compact_days


def infer_weekly_plan(profile, routine_hits, reranked_products):
    """Function to build the main routine."""
    used = set()
    
    # pick the main products
    cleanser = choose_best_product_by_type(reranked_products, 'cleanser', used)
    if cleanser:
        used.add((cleanser['source_type'], cleanser['row_id']))
    serum = choose_best_product_by_type(reranked_products, 'serum', used)
    if serum:
        used.add((serum['source_type'], serum['row_id']))
    moisturizer = choose_best_product_by_type(reranked_products, 'moisturizer', used)
    if moisturizer:
        used.add((moisturizer['source_type'], moisturizer['row_id']))
    sunscreen = choose_best_product_by_type(reranked_products, 'sunscreen', used)
    if sunscreen:
        used.add((sunscreen['source_type'], sunscreen['row_id']))
    mask = choose_best_product_by_type(reranked_products, 'mask', used)

    # read the skin profile
    concerns = set(profile.get('concerns', []))
    skin_types = set(profile.get('skin_types', []))

    # build evening and morning routine
    evening_serum_days = []
    if serum:
        if 'sensible Haut' in skin_types or 'rötungen' in concerns:
            evening_serum_days = ['Mo', 'Mi', 'Fr']
        else:
            evening_serum_days = ['Mo', 'Di', 'Do', 'Sa']

    mask_days = ['So'] if mask else []

    base_morning = []
    if cleanser:
        base_morning.append({'step': 'Reinigung', 'product': cleanser})
    if serum and ('trockenheit' in concerns or 'pigmentflecken' in concerns):
        base_morning.append({'step': 'Serum', 'product': serum})
    if moisturizer:
        base_morning.append({'step': 'Feuchtigkeitspflege', 'product': moisturizer})
    if sunscreen:
        base_morning.append({'step': 'Sonnenschutz', 'product': sunscreen})

    base_evening = []
    if cleanser:
        base_evening.append({'step': 'Reinigung', 'product': cleanser})
    base_evening_tail = [{'step': 'Feuchtigkeitspflege', 'product': moisturizer}] if moisturizer else []

    # build the full week
    weekly_plan = []
    day_names = ['Mo', 'Di', 'Mi', 'Do', 'Fr', 'Sa', 'So']
    for day in day_names:
        morning_steps = list(base_morning)
        evening_steps = list(base_evening)
        if serum and day in evening_serum_days:
            evening_steps.append({'step': 'Serum', 'product': serum})
        if mask and day in mask_days:
            evening_steps.append({'step': 'Maske', 'product': mask})
        evening_steps.extend(base_evening_tail)
        weekly_plan.append({'day': day, 'morning': morning_steps, 'evening': evening_steps})

    # prepare OCR context
    # convert OCR routine hits into a compatct routine_context list 
    routine_context = [
        {'file_name': r['meta'].get('file_name', ''), 'ocr_text': r['meta'].get('ocr_text', '')}
        for r in routine_hits
    ]

    return {
        'weekly_plan': weekly_plan,
        'weekly_plan_compact': summarize_weekly_plan(weekly_plan),
        'routine_context': routine_context,
    }


## Functions to prepare the final prompt for the language model and make sure it stays within the token budget

In [8]:
def format_context_blocks_compact(ingredient_hits, routine_hits, reranked_products, ingredient_limit=max_ingredient_hits, routine_limit=max_routine_hits, product_limit=max_product_hits):
    """This function compresses the retrieved RAG context into small, structured blocks.
    It takes the top ingredient hits, routine hits and reranked products, then keeps only a few fields from each one."""
    ingredient_context = []
    for hit in ingredient_hits[:ingredient_limit]:
        m = hit['meta']
        ingredient_context.append({
            'ingredient': clean_text(m.get('ingredient', '')),
            'topic': clean_text(m.get('topic', '')),
            'text': shorten(m.get('text', ''), max_ingredient_text_len),
        })

    routine_context = []
    for hit in routine_hits[:routine_limit]:
        routine_context.append({
            'file_name': hit['meta'].get('file_name', ''),
            'ocr_text': shorten(hit['meta'].get('ocr_text', ''), max_routine_text_len),
        })

    product_context = []
    for item in reranked_products[:product_limit]:
        m = item['meta']
        product_context.append({
            'brand': clean_text(m.get('brand', '')),
            'name': clean_text(m.get('name', '')),
            'price': clean_text(m.get('price', '')),
            'product_type': item.get('product_type', 'other'),
            'instruction': shorten(m.get('instruction', ''), max_product_instruction_len),
            'ingredients': shorten(m.get('ingredients', ''), max_product_ingredients_len),
            'warnings': shorten(m.get('warnings', ''), max_product_warnings_len),
        })

    return {'ingredients': ingredient_context, 'routines': routine_context, 'products': product_context}


def estimate_tokens(text: str):
    """This is a rough token estimator. It cleans the text first, then estimates token count."""
    text = clean_text(text)
    if not text:
        return 0
    return max(1, int(len(text) / 4))


def build_generation_payload(user_text, profile, inferred_routine, context):
    """This function builds the actual prompt that will be sent to the model."""
    system_prompt = '''
Du bist ein deutschsprachiger Skincare-RAG-Assistent.
Nutze ausschließlich den bereitgestellten Kontext.
Wenn etwas im Kontext nicht ausreichend belegt ist, sage das offen.
Keine medizinischen Diagnosen.
Antworte knapp, strukturiert und in sauberem Markdown.
Struktur:
1. Empfohlene Routine
2. Erklärte Wirkstoffe
3. Produktempfehlungen mit Erklärung bei was sie helfen
'''.strip()

    # create a smaller version of the weekly plan
    compact_routine = {
        'weekly_plan': inferred_routine.get('weekly_plan_compact', []),
        'routine_context': [
            {
                'file_name': clean_text(item.get('file_name', '')),
                'ocr_text': shorten(item.get('ocr_text', ''), max_routine_text_len),
            }
            for item in inferred_routine.get('routine_context', [])[:1]
        ]
    }

    user_prompt = f'''Nutzeranfrage:
{user_text}

Erkanntes Hautprofil:
{json.dumps(profile, ensure_ascii=False, separators=(',', ':'))}

Vorstrukturierte Wochenroutine:
{json.dumps(compact_routine, ensure_ascii=False, separators=(',', ':'))}

Abgerufener RAG-Kontext:
{json.dumps(context, ensure_ascii=False, separators=(',', ':'))}

Erzeuge jetzt die finale Antwort.
'''.strip()

    estimated_input_tokens = estimate_tokens(system_prompt) + estimate_tokens(user_prompt)
    return {'system_prompt': system_prompt, 'user_prompt': user_prompt, 'estimated_input_tokens': estimated_input_tokens}


def trim_context_to_budget(user_text, profile, inferred_routine, ingredient_hits, routine_hits, reranked_products):
    """This function makes sure the final prompt does not exceed the context limit."""
    ingredient_limit = max_ingredient_hits
    routine_limit = max_routine_hits
    product_limit = max_product_hits

    while True:
        # build a compact context with current limits
        context = format_context_blocks_compact(
            ingredient_hits,
            routine_hits,
            reranked_products,
            ingredient_limit=ingredient_limit,
            routine_limit=routine_limit,
            product_limit=product_limit,
        )
        # build the prompt payload from that context
        payload = build_generation_payload(user_text, profile, inferred_routine, context)
        budget = max_context_tokens - target_response_tokens - token_safety_margin
        
        # check whether the estimated input tokens fit within the budget
        if payload['estimated_input_tokens'] <= budget:
            return {'context': context, 'payload': payload, 'limits': {'ingredient_limit': ingredient_limit, 'routine_limit': routine_limit, 'product_limit': product_limit}}

        if product_limit > 1:
            product_limit -= 1
            continue
        if ingredient_limit > 1:
            ingredient_limit -= 1
            continue
        if routine_limit > 0:
            routine_limit -= 1
            continue

        return {'context': context, 'payload': payload, 'limits': {'ingredient_limit': ingredient_limit, 'routine_limit': routine_limit, 'product_limit': product_limit}}


In [9]:
def get_lmstudio_client(base_url):
    return OpenAI(base_url=base_url, api_key=lmstudio_api_key)


def test_lmstudio_connection(base_url=lmstudio_base_url):
    client = get_lmstudio_client(base_url)
    return client.models.list()


def generate_rag_answer_lmstudio(user_text, profile, inferred_routine, ingredient_hits, routine_hits, reranked_products, model_name, base_url):
    client = get_lmstudio_client(base_url)
    optimized = trim_context_to_budget(user_text, profile, inferred_routine, ingredient_hits, routine_hits, reranked_products)
    context = optimized['context']
    payload = optimized['payload']
    print('Verwendete Limits:', optimized['limits'])
    print('Geschätzte Input-Tokens:', payload['estimated_input_tokens'])
    print('Geplante Antwort-Tokens:', target_response_tokens)
    print('Maximales Kontextbudget:', max_context_tokens)
    completion = client.chat.completions.create(
        model=model_name,
        temperature=0.2,
        max_tokens=target_response_tokens,
        messages=[
            {'role': 'system', 'content': payload['system_prompt']},
            {'role': 'user', 'content': payload['user_prompt']},
        ],
    )
    return {'answer': completion.choices[0].message.content, 'optimized_context': context, 'payload': payload, 'limits': optimized['limits']}

## Prepare the whole chatbot system

In [10]:
def prepare_system():
    # load multilingual embedding model used for semantic search
    model = SentenceTransformer(embed_model)

    # read product csv and turn each product into a document
    df, product_docs = load_products(products_csv)

    # load the ingredient knowledge json and convert it into documents
    ingredient_docs = load_ingredient_knowledge(INGREDIENTS_JSON)

    # OCR the routine images and record any missing or failed files
    routine_docs, missing_files, failed_files = load_routine_images(ROUTINE_IMAGE_PATHS)

    # build a vector index for product documents
    product_index = build_index(product_docs, model)

    # build a vector index for ingredient documents
    ingredient_index = build_index(ingredient_docs, model)

    # build a vector index for OCR routine documents
    routine_index = build_index(routine_docs, model)
    return {
        'df': df,
        'model': model,
        'product_docs': product_docs,
        'ingredient_docs': ingredient_docs,
        'routine_docs': routine_docs,
        'missing_files': missing_files,
        'failed_files': failed_files,
        'product_index': product_index,
        'ingredient_index': ingredient_index,
        'routine_index': routine_index,
    }


In [17]:
system = prepare_system()
print('Products:', len(system['product_docs']))
print('Ingredient documents:', len(system['ingredient_docs']))
print('Routine OCR documents:', len(system['routine_docs']))
if system['missing_files']:
    print('\nMissing OCR files:')
    for item in system['missing_files']:
        print('-', item)
if system['failed_files']:
    print('\nOCR failures:')
    for item in system['failed_files']:
        print('-', item)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4154.47it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Products: 1119
Ingredient documents: 7
Routine OCR documents: 7


In [ ]:
# check lmstudio connection
# models_info = test_lmstudio_connection()
# models_info


## Enter user input

In [18]:
user_text = 'Ich habe trockene Haut, Rötungen und Pickel. Entwickle eine Skincare Routine dafür.'
user_text


'Ich habe trockene Haut, Rötungen und Pickel. Entwickle eine Skincare Routine dafür.'

In [19]:
# detect skin profile from the text
profile = extract_skin_profile(user_text)

# search the ingredient knowledge base for relevant ingredient explanations
ingredient_hits = semantic_search(user_text, system['ingredient_docs'], system['ingredient_index'], system['model'], top_k=5)

# search OCR routine examples
routine_hits = semantic_search(user_text, system['routine_docs'], system['routine_index'], system['model'], top_k=4)

# search product database
product_hits = semantic_search(user_text, system['product_docs'], system['product_index'], system['model'], top_k=20)

# reorder the product hits using skincare logic
reranked_products = rerank_products(product_hits, profile, max_items=8)

# build a weekly skincare routine
inferred_routine = infer_weekly_plan(profile, routine_hits, reranked_products)


In [20]:
# compress the context so the prompt stays within token limits
optimized = trim_context_to_budget(
    user_text=user_text,
    profile=profile,
    inferred_routine=inferred_routine,
    ingredient_hits=ingredient_hits,
    routine_hits=routine_hits,
    reranked_products=reranked_products,
)

optimized['limits'], optimized['payload']['estimated_input_tokens']


({'ingredient_limit': 2, 'routine_limit': 1, 'product_limit': 3}, 2151)

## Send final prompt to the local model
The model generates the final skincare answer

In [21]:
result = generate_rag_answer_lmstudio(
    user_text=user_text,
    profile=profile,
    inferred_routine=inferred_routine,
    ingredient_hits=ingredient_hits,
    routine_hits=routine_hits,
    reranked_products=reranked_products,
    model_name=lmstudio_model,
    base_url=lmstudio_base_url,
)

answer = result['answer']
display(Markdown(answer))


Verwendete Limits: {'ingredient_limit': 2, 'routine_limit': 1, 'product_limit': 3}
Geschätzte Input-Tokens: 2151
Geplante Antwort-Tokens: 500
Maximales Kontextbudget: 3200


### 1. Empfohlene Routine

**Montag bis Freitag:**
- **Morgens:**
  - Reinigung: CeraVe Feuchtigkeitsspendende Gesichtscreme (Artikelpreis 14,99 €)
  - Serum: ISANA Korean Skincare Power Serum (Artikelpreis 5,99 €)
  - Feuchtigkeitspflege: Cetaphil Feuchtigkeitscreme (Artikelpreis 9,49 €)

- **Abends:**
  - Reinigung: CeraVe Feuchtigkeitsspendende Gesichtscreme (Artikelpreis 14,99 €)
  - Serum: ISANA Korean Skincare Power Serum (Artikelpreis 5,99 €)
  - Feuchtigkeitspflege: Cetaphil Feuchtigkeitscreme (Artikelpreis 9,49 €)

**Samstag und Sonntag:**
- **Morgens:**
  - Reinigung: CeraVe Feuchtigkeitsspendende Gesichtscreme (Artikelpreis 14,99 €)
  - Serum: ISANA Korean Skincare Power Serum (Artikelpreis 5,99 €)
  - Feuchtigkeitspflege: Cetaphil Feuchtigkeitscreme (Artikelpreis 9,49 €)

- **Abends:**
  - Reinigung: CeraVe Feuchtigkeitsspendende Gesichtscreme (Artikelpreis 14,99 €)
  - Feuchtigkeitspflege: Cetaphil Feuchtigkeitscreme (Artikelpreis 9,49 €)

### 2. Erklärte Wirkstoffe

- **Peptide:** In der Routine sind sie nicht direkt enthalten, aber sie helfen bei der Förmung von Hautstruktur und können die Haut festiger machen.
- **Azelainsäure:** Enthalten in dem ISANA Serum, sie beruhigt Rötungen und ist besonders wirksam auf trockene und problematische Haut.

### 3. Produktempfehlungen

**CeraVe Feuchtigkeitsspendende Gesicht

## Write everything into a function to try different user texts
I want to conduct experiments on different skin types and skin problems to evaluate the model's answer.

In [22]:
def ask_skincare_assistant(user_text: str, system_state=None, top_k_ingredients=5, top_k_routines=4, top_k_products=20):
    if not user_text or not user_text.strip():
        raise ValueError("user_text darf nicht leer sein.")

    if system_state is None:
        system_state = system

    profile = extract_skin_profile(user_text) if "extract_skin_profile" in globals() else extract_skin_profile(user_text)

    ingredient_hits = semantic_search(
        user_text,
        system_state["ingredient_docs"],
        system_state["ingredient_index"],
        system_state["model"],
        top_k=top_k_ingredients,
    )

    routine_hits = semantic_search(
        user_text,
        system_state["routine_docs"],
        system_state["routine_index"],
        system_state["model"],
        top_k=top_k_routines,
    )

    product_hits = semantic_search(
        user_text,
        system_state["product_docs"],
        system_state["product_index"],
        system_state["model"],
        top_k=top_k_products,
    )

    reranked_products = rerank_products(product_hits, profile, max_items=8)
    inferred_routine = infer_weekly_plan(profile, routine_hits, reranked_products)

    result = generate_rag_answer_lmstudio(
        user_text=user_text,
        profile=profile,
        inferred_routine=inferred_routine,
        ingredient_hits=ingredient_hits,
        routine_hits=routine_hits,
        reranked_products=reranked_products,
        model_name=lmstudio_model,
        base_url=lmstudio_base_url,
    )

    return {
        "user_text": user_text,
        "profile": profile,
        "ingredient_hits": ingredient_hits,
        "routine_hits": routine_hits,
        "product_hits": product_hits,
        "reranked_products": reranked_products,
        "inferred_routine": inferred_routine,
        "answer": result["answer"],
        "context": result["optimized_context"],
        "payload": result["payload"],
        "limits": result["limits"],
    }

def ask(user_text: str) -> str:
    result = ask_skincare_assistant(user_text)
    return result["answer"]

## Evaluate test cases
I execute everything in a different cell so make sure nothing could crash in the middle of a loop.

In [23]:
test_cases = [
    "Ich habe trockene Haut, Rötungen und Pickel. Entwickle eine Skincare Routine dafür.",
    "Ich habe ölige Haut und suche eine Routine gegen Unreinheiten.",
    "Ich habe sensible Haut mit Pigmentflecken und brauche eine milde Routine.",
    "Ich habe Mischhaut und möchte etwas gegen verstopfte Poren.",
    "Ich suche eine Anti-Aging-Routine bei trockener Haut."
]


In [24]:
display(Markdown(ask(test_cases[0])))

Verwendete Limits: {'ingredient_limit': 2, 'routine_limit': 1, 'product_limit': 3}
Geschätzte Input-Tokens: 2151
Geplante Antwort-Tokens: 500
Maximales Kontextbudget: 3200


### 1. Empfohlene Routine

**Montag - Sonntag (Morgens & Abends):**
- **Reinigung:**
  - **CeraVe Feuchtigkeitsspendende Gesichtscreme** (Artikelpreis 14,99 €)
    - Wirkstoffe: Synthetische Peptide, Niacinamide
    - Vorteile: Stabil und geeignet für problematische Haut.
- **Serum:**
  - **ISANA Korean Skincare Power Serum** (Artikelpreis 5,99 €)
    - Wirkstoffe: Azelainsäure
    - Vorteile: Beruhigt Rötungen und verbessert die Hautstruktur.
- **Feuchtigkeitspflege:**
  - **Cetaphil Feuchtigkeitscreme** (Artikelpreis 9,49 €)
    - Wirkstoffe: Glycerin, Dimethicone
    - Vorteile: Tägliche Pflege für trockene und problematische Haut.

**Samstag & Sonntag (nur Abends):**
- **Reinigung:**
  - **CeraVe Feuchtigkeitsspendende Gesichtscreme** (Artikelpreis 14,99 €)
- **Feuchtigkeitspflege:**
  - **Cetaphil Feuchtigkeitscreme** (Artikelpreis 9,49 €)

### 2. Erklärte Wirkstoffe

- **Peptide:** 
  - **Funktion:** Stabilisieren die Haut und verbessern die Hautstruktur.
  - **Wirkung:** Erhöhen die Fehldeckerfähigkeit der Haut, was zu einer besseren Haltbarkeit der Hautzellen führt.

- **Azelainsäure:**
  - **Funktion:** Beruhigt Rötungen und verbessert die Hautstruktur.
  - **Wirkung:** Stabilisiert die Hautbarriere und reduziert Entzündungsreaktionen, was zu einer besseren Behandlung von Rötungen führt.

### 3. Produktempfehlungen

- **CeraVe Feuchtigkeitsspend

In [25]:
display(Markdown(ask(test_cases[1])))

Verwendete Limits: {'ingredient_limit': 2, 'routine_limit': 1, 'product_limit': 3}
Geschätzte Input-Tokens: 1979
Geplante Antwort-Tokens: 500
Maximales Kontextbudget: 3200


1. **Empfohlene Routine:**
   - **Morgens:**
     1. Reinigung: CeraVe Schäumendes Reinigungsgel (Artikelpreis 13,99 €)
        - In der Hand mit etwas Wasser aufschäumen und sanft auf die Haut aufmassieren. Sorgfältig abspülen.
     2. Feuchtigkeitspflege: Schaebens Reine Haut Anti-Pickel Patches (Artikelpreis 1,29 €)
        - Gesicht reinigen und einen Patch mit sauberen, trockenen Händen von der Folie abziehen. Den Patch direkt auf die Haut legen.
   - **Abends:**
     1. Reinigung: CeraVe Schäumendes Reinigungsgel (Artikelpreis 13,99 €)
        - In der Hand mit etwas Wasser aufschäumen und sanft auf die Haut aufmassieren. Sorgfältig abspülen.
     2. Serum: Judith Williams Cosmetics Peptide+ Gesichtsserum (Artikelpreis 17,99 €)
        - Morgens und/oder abends auf die gereinigte Haut geben und sanft einmassieren.
     3. Feuchtigkeitspflege: Schaebens Reine Haut Anti-Pickel Patches (Artikelpreis 1,29 €)
        - Gesicht reinigen und einen Patch mit sauberen, trockenen Händen von der Folie abziehen. Den Patch direkt auf die Haut legen.

2. **Erklärte Wirkstoffe:**
   - **Peptide:** Stabilisieren die Hautstruktur und verbessern die Fehlend festigkeit.
   - **Azelainsäure:** Beruhigt Rötungen, bietet Schutz vor Sonnenbrand und reduziert Unreinheiten.

3. **Produktempfehlungen:**
   - **CeraVe Schäumendes Reinigungsgel (Artikelpreis 13,99 €):** Reinhält Azelainsäure, die die Haut beruhigt und Unreinheiten reduziert.
   - **Judith Williams Cos

In [26]:
display(Markdown(ask(test_cases[2])))

Verwendete Limits: {'ingredient_limit': 2, 'routine_limit': 1, 'product_limit': 3}
Geschätzte Input-Tokens: 1762
Geplante Antwort-Tokens: 500
Maximales Kontextbudget: 3200


1. **Empfohlene Routine:**
   - **Montag bis Freitag (Morgens):**
     - Reinigung: DAYTOX Peptide Booster Peeling (Artikelpreis 11,99 €)
       - Vor Gebrauch gut schütteln. Morgens und abends nach der Reinigung 3-4 Tropfen auf die gereinigte Haut anwenden.
     - Feuchtigkeitspflege: Cetaphil Feuchtigkeitscreme (Artikelpreis 9,49 €)
       - Zur täglichen Pflege geeignet. Anwenden morgens & abends.
   - **Samstag und Sonntag (Morgens):**
     - Reinigung: DAYTOX Peptide Booster Peeling (Artikelpreis 11,99 €)
       - Vor Gebrauch gut schütteln. Morgens und abends nach der Reinigung 3-4 Tropfen auf die gereinigte Haut anwenden.
     - Feuchtigkeitspflege: Cetaphil Feuchtigkeitscreme (Artikelpreis 9,49 €)
       - Zur täglichen Pflege geeignet. Anwenden morgens & abends.

2. **Erklärte Wirkstoffe:**
   - **Peptide:** Stabilisieren die Haut und helfen bei der Fehlendem Festigkeit.
   - **Niacinamid (Vitamine B3):** Wirkt für alle Hauttypen, unabhängig von den Problemen. Erhilft bei Vergrößerung von Poren.

3. **Produktempfehlungen:**
   - **DAYTOX Peptide Booster Peeling:** Hilft bei der Fehlendem Festigkeit und ist besonders geeignet für sensible Haut.
   - **Cetaphil Feuchtigkeitscreme:** Eignet sich gut zur täglichen Pflege von sensibler Haut.

In [27]:
display(Markdown(ask(test_cases[3])))

Verwendete Limits: {'ingredient_limit': 2, 'routine_limit': 1, 'product_limit': 3}
Geschätzte Input-Tokens: 1911
Geplante Antwort-Tokens: 500
Maximales Kontextbudget: 3200


### 1. Empfohlene Routine

**Montag bis Freitag:**
- Morgens und Abends:
  - Reinigung: **Neutrogena Anti-Pickel Pflegeset** (Artikelpreis 8,49 €)
  - Feuchtigkeitspflege: **Neutrogena Anti-Pickel Tägliche Feuchtigkeitspflege** (Artikelpreis 4,49 €)

**Samstag und Sonntag:**
- Morgens:
  - Reinigung: **Neutrogena Anti-Pickel Pflegeset** (Artikelpreis 8,49 €)
  - Feuchtigkeitspflege: **Neutrogena Anti-Pickel Tägliche Feuchtigkeitspflege** (Artikelpreis 4,49 €)
- Abends:
  - Reinigung: **Neutrogena Anti-Pickel Pflegeset** (Artikelpreis 8,49 €)
  - Maske: **Schaebens Panthenol 3-Zonen Tuchmaske** (Artikelpreis 1,79 €)
  - Feuchtigkeitspflege: **Neutrogena Anti-Pickel Tägliche Feuchtigkeitspflege** (Artikelpreis 4,49 €)

### 2. Erklärte Wirkstoffe

- **Azelainsäure**: Ein echtes Multitalent, das nicht nur bei Rötungen hilfreich ist, sondern auch bei der Verbesserung des Hautausbaus und der Reduzierung von verstopften Poren wirkt.
- **Niacinamid (Vitamine B3)**: Wirksam für alle Hauttypen und Probleme, einschließlich vergrößerter Poren. Er hilft, die Hautstruktur zu verbessern und die Porengröße zu reduzieren.

### 3. Produktempfehlungen

- **Neutrogena Anti-Pickel Pflegeset** (Artikelpreis 8,49 €): Reinigt das Gesicht effektiv und enthält Azelainsäure, die verstopfte Poren reduziert.
- **Schaebens Pan

In [28]:
display(Markdown(ask(test_cases[4])))

Verwendete Limits: {'ingredient_limit': 2, 'routine_limit': 1, 'product_limit': 3}
Geschätzte Input-Tokens: 2443
Geplante Antwort-Tokens: 500
Maximales Kontextbudget: 3200


### 1. Empfohlene Routine

**Morgens:**
1. **Reinigung:** Hormocenta Anti-Age Nachtcreme (Artikelpreis 5,99 €)
2. **Serum:** L’Oréal Paris Kollagen Experte Anti-Age Serum (Artikelpreis 9,99 €)
3. **Feuchtigkeitspflege:** L’Oréal Paris Age Perfect Tagescreme für reife und straffere Haut, gegen Altersflecken (Artikelpreis 9,99 €)

**Abends:**
1. **Reinigung:** Hormocenta Anti-Age Nachtcreme (Artikelpreis 5,99 €)
2. **Serum:** L’Oréal Paris Kollagen Experte Anti-Age Serum (Artikelpreis 9,99 €)
3. **Feuchtigkeitspflege:** L’Oréal Paris Age Perfect Tagescreme für reife und straffere Haut, gegen Altersflecken (Artikelpreis 9,99 €)

**Sonntags:**
1. **Reinigung:** Hormocenta Anti-Age Nachtcreme (Artikelpreis 5,99 €)
2. **Serum:** L’Oréal Paris Kollagen Experte Anti-Age Serum (Artikelpreis 9,99 €)
3. **Feuchtigkeitspflege:** L’Oréal Paris Age Perfect Tagescreme für reife und straffere Haut, gegen Altersflecken (Artikelpreis 9,99 €)

### 2. Erklärte Wirkstoffe

- **Peptide:** Stabilisieren die Hautstruktur und fördern die Synthese von Elastin und Collagen.
- **Niacinamid:** Verbesserung der Textur und Verringerung großer Poren.

### 3. Produktempfehlungen mit Erklärung

**Hormocenta Anti-Age Nachtcreme (Artikelpreis 5,99 €):**
- **Wirkstoffe:** Keine spezifischen Wirkstoffe sind in der Beschreibung aufgeführt.
- **Zweck:** Reinigung des Gesicht